# 10 - Prediction Pipeline

### Objective

Build a reusable prediction pipeline that loads the saved models and predicts the next-day stock direction with prediction probability.

In [6]:
import joblib
import pandas as pd
from pathlib import Path

In [2]:
stocks = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "RELIANCE.NS",
    "TCS.NS",
    "HDFCBANK.NS",
    "INFY.NS"
]

## Model Features

Define the same features and feature order that were used during model training.

In [3]:
features = [
    "Daily_Return",
    "Return_Lag_1",
    "Return_Lag_2",
    "Return_Lag_3",
    "SMA_5",
    "SMA_20",
    "SMA_50",
    "EMA_20",
    "Volatility_20",
    "Volume_Change",
    "High_Low_Range",
    "Open_Close_Change",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Hist"
]

## Load Final Models

Load the final trained model for each stock from the saved model files.

In [7]:
models = {}

model_dir = Path("../models/final")

for stock in stocks:
    
    model_path = model_dir / f"{stock}_model.joblib"
    
    models[stock] = joblib.load(model_path)
    
    print(f"{stock} model loded.")

AAPL model loded.
MSFT model loded.
NVDA model loded.
AMZN model loded.
RELIANCE.NS model loded.
TCS.NS model loded.
HDFCBANK.NS model loded.
INFY.NS model loded.


## Load Feature Data

Load the feature-engineered datasets used to prepare the latest observation for prediction.

In [9]:
processed_data = {}

for stock in stocks:
    
    df = pd.read_csv(f"../data/processed/{stock}_features.csv", parse_dates=['Date'])
    df = df.sort_values("Date").reset_index(drop=True)
    
    processed_data[stock] = df

## Prepare Latest Observation

Extract the most recent feature values for each stock in the same format used during model training.

In [10]:
latest_features = {}

for stock in stocks:
    
    df = processed_data[stock]
    
    latest_row = df[features].iloc[[-1]]
    
    latest_features[stock] = latest_row

In [11]:
latest_features["AAPL"]

,Daily_Return,Return_Lag_1,Return_Lag_2,Return_Lag_3,SMA_5,SMA_20,SMA_50,EMA_20,Volatility_20,Volume_Change,High_Low_Range,Open_Close_Change,RSI_14,MACD,MACD_Signal,MACD_Hist
2764,-0.002484,0.001317,-0.001497,0.005324,272.775208,275.676814,271.88713,273.689884,0.006813,-0.066438,0.006591,0.00099,39.130336,0.078971,0.808691,-0.72972


In [12]:
latest_features["AAPL"].shape

(1, 16)

In [13]:
latest_features["AAPL"].isnull().sum()

Daily_Return         0
Return_Lag_1         0
Return_Lag_2         0
Return_Lag_3         0
SMA_5                0
SMA_20               0
SMA_50               0
EMA_20               0
Volatility_20        0
Volume_Change        0
High_Low_Range       0
Open_Close_Change    0
RSI_14               0
MACD                 0
MACD_Signal          0
MACD_Hist            0
dtype: int64

## Prediction Function

Create a reusable function that uses the selected stock's saved model and latest feature values to predict the next-day direction and its probability.

In [23]:
def predict_stock(stock):
    
    # get model
    model = models[stock]
    
    # get latest features
    X_latest = latest_features[stock]
    
    # predict direction
    prediction = model.predict(X_latest)[0]
    
    # predict probabilites
    probabilities = model.predict_proba(X_latest)[0]
    
    # Probability of predicted class
    confidence = probabilities[prediction]
    
    # Latest date
    latest_date = processed_data[stock]["Date"].iloc[-1]

    # Convert 0/1 into readable label
    direction = "UP" if prediction == 1 else "DOWN"

    dir = {
        "Stock": stock,
        "Date": latest_date,
        "Prediction": direction,
        "Probability": round(float(confidence) * 100, 2)
    }
    return dir


## Predictions for All Stocks

Generate next-day direction predictions and probabilities for all stocks using their final saved models.

In [32]:
all_predictions = []

for stock in stocks:
    result = predict_stock(stock=stock)
    all_predictions.append(result)

predictions_df = pd.DataFrame(all_predictions)

model_selection = pd.read_csv("../results/final_model_selection.csv")

predictions_df = predictions_df.merge(
    model_selection[["Stock", "Best Model"]],
    on="Stock",
    how="left"
)

predictions_df

,Stock,Date,Prediction,Probability,Best Model
0,AAPL,2025-12-30,UP,51.44,Logistic Regression
1,MSFT,2025-12-30,UP,52.28,Logistic Regression
2,NVDA,2025-12-30,DOWN,54.00,Default Random Forest
3,AMZN,2025-12-30,DOWN,50.52,Logistic Regression
4,RELIANCE.NS,2025-12-30,DOWN,53.00,Default Random Forest
5,TCS.NS,2025-12-30,DOWN,58.00,Default Random Forest
6,HDFCBANK.NS,2025-12-30,DOWN,64.52,Tuned Random Forest
7,INFY.NS,2025-12-30,UP,54.70,Logistic Regression


## Conclusion

A reusable prediction pipeline was created to load the final models and generate next-day stock direction predictions with probabilities.

The pipeline supports all eight stocks and is ready to be integrated into the StockVision Streamlit dashboard.